# IPL Auction Price Prediction — Improved Regression

### Objective
Predict IPL player auction sold prices using player characteristics and cricket performance statistics.

### Models
1. Linear Regression — baseline
2. Ridge Regression — regularized improved model

### Additional techniques
- Exploratory Data Analysis
- Missing-value treatment
- Categorical encoding
- IQR-based outlier capping on training features
- Standardization
- Log transformation of the target
- Ridge regularization
- Cross-validation for selecting Ridge alpha
- Residual analysis
- MAE, RMSE and R²
- Feature importance through standardized Ridge coefficients

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the dataset

In [ ]:
df = pd.read_csv("IPL IMB381IPL2013.csv")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())

## 3. Understand the data

In [ ]:
df.info()

In [ ]:
print("Missing values:")
display(df.isnull().sum())

In [ ]:
display(df.describe())

## 4. Target variable — SOLD PRICE

In [ ]:
display(df["SOLD PRICE"].describe())

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["SOLD PRICE"], bins=20)
plt.xlabel("Sold Price")
plt.ylabel("Number of Players")
plt.title("Distribution of IPL Auction Prices")
plt.show()

## 5. Exploratory Data Analysis

In [ ]:
role_price = df.groupby("PLAYING ROLE")["SOLD PRICE"].mean().sort_values(ascending=False)
display(role_price)

In [ ]:
plt.figure(figsize=(8,5))
role_price.plot(kind="bar")
plt.xlabel("Playing Role")
plt.ylabel("Average Sold Price")
plt.title("Average Auction Price by Playing Role")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["RUNS-S"], df["SOLD PRICE"])
plt.xlabel("IPL Runs")
plt.ylabel("Sold Price")
plt.title("IPL Runs vs Auction Price")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["WKTS"], df["SOLD PRICE"])
plt.xlabel("IPL Wickets")
plt.ylabel("Sold Price")
plt.title("IPL Wickets vs Auction Price")
plt.show()

## 6. Select features

Excluded:
- `PLAYER NAME` — identifier
- `TEAM` — related to the auction outcome
- `BASE PRICE` — excluded so the model does not directly use the auction starting price
- `SOLD PRICE` — target

The model uses player characteristics and cricket performance variables.

In [ ]:
features = [
    "AGE",
    "COUNTRY",
    "PLAYING ROLE",
    "T-RUNS",
    "T-WKTS",
    "ODI-RUNS-S",
    "ODI-SR-B",
    "ODI-WKTS",
    "ODI-SR-BL",
    "CAPTAINCY EXP",
    "RUNS-S",
    "HS",
    "AVE",
    "SR-B",
    "SIXERS",
    "RUNS-C",
    "WKTS",
    "AVE-BL",
    "ECON",
    "SR-BL"
]

X = df[features].copy()
y = df["SOLD PRICE"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

## 7. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 8. Missing-value treatment

Missing-value statistics are calculated from the training data and then applied to both training and test data. This avoids using information from the test set during preprocessing.

In [ ]:
numeric_columns = X_train.select_dtypes(include=np.number).columns
categorical_columns = X_train.select_dtypes(include="object").columns

X_train = X_train.copy()
X_test = X_test.copy()

for col in numeric_columns:
    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

for col in categorical_columns:
    mode_value = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_value)
    X_test[col] = X_test[col].fillna(mode_value)

print("Remaining missing values in training data:", X_train.isnull().sum().sum())
print("Remaining missing values in test data:", X_test.isnull().sum().sum())

## 9. Categorical encoding

Convert `COUNTRY` and `PLAYING ROLE` into dummy variables.

In [ ]:
X_train = pd.get_dummies(
    X_train,
    columns=["COUNTRY", "PLAYING ROLE"],
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=["COUNTRY", "PLAYING ROLE"],
    drop_first=True
)

# Ensure both datasets have exactly the same columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Training shape after encoding:", X_train.shape)
print("Testing shape after encoding:", X_test.shape)

## 10. Outlier treatment using IQR

We will **cap extreme values in the predictor variables** rather than deleting observations.

This is safer for our small dataset because genuine high-performing players can naturally have unusually high statistics.

The IQR rule uses:

- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

The bounds are calculated using only the training data.

In [ ]:
numeric_encoded = X_train.select_dtypes(include=np.number).columns

lower_bounds = {}
upper_bounds = {}

for col in numeric_encoded:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bounds[col] = Q1 - 1.5 * IQR
    upper_bounds[col] = Q3 + 1.5 * IQR

for col in numeric_encoded:
    X_train[col] = X_train[col].clip(
        lower_bounds[col],
        upper_bounds[col]
    )

    X_test[col] = X_test[col].clip(
        lower_bounds[col],
        upper_bounds[col]
    )

print("Outlier capping completed.")

## 11. Log-transform the target

Auction prices are highly right-skewed.

We use:

`log(1 + SOLD PRICE)`

This reduces the effect of extreme target values and can make the regression relationship more stable.

In [ ]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

plt.figure(figsize=(8,5))
plt.hist(y_train_log, bins=20)
plt.xlabel("Log(Sold Price + 1)")
plt.ylabel("Number of Players")
plt.title("Log-Transformed Auction Price")
plt.show()

## 12. Standardize features

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 13. Baseline Linear Regression

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train_scaled, y_train_log)

linear_pred_log = linear_model.predict(X_test_scaled)

# Convert predictions back to original price scale
linear_pred = np.expm1(linear_pred_log)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression")
print("-----------------")
print("MAE :", round(linear_mae, 2))
print("RMSE:", round(linear_rmse, 2))
print("R²  :", round(linear_r2, 3))

## 14. Ridge Regression with cross-validation

Ridge regression adds regularization to reduce the effect of multicollinearity and overly large coefficients.

We use cross-validation to select the best `alpha`.

In [ ]:
ridge = Ridge()

alpha_values = {
    "alpha": [0.01, 0.1, 1, 10, 100, 1000]
}

ridge_search = GridSearchCV(
    ridge,
    alpha_values,
    cv=5,
    scoring="neg_mean_squared_error"
)

ridge_search.fit(X_train_scaled, y_train_log)

print("Best alpha:", ridge_search.best_params_["alpha"])

In [ ]:
ridge_model = ridge_search.best_estimator_

ridge_pred_log = ridge_model.predict(X_test_scaled)

# Convert predictions back to original price scale
ridge_pred = np.expm1(ridge_pred_log)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

print("Ridge Regression")
print("----------------")
print("MAE :", round(ridge_mae, 2))
print("RMSE:", round(ridge_rmse, 2))
print("R²  :", round(ridge_r2, 3))

## 15. Compare the models

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression"],
    "MAE": [linear_mae, ridge_mae],
    "RMSE": [linear_rmse, ridge_rmse],
    "R2": [linear_r2, ridge_r2]
})

display(results)

## 16. Select the better model

In [ ]:
if ridge_r2 >= linear_r2:
    best_model_name = "Ridge Regression"
    best_predictions = ridge_pred
else:
    best_model_name = "Linear Regression"
    best_predictions = linear_pred

print("Best model based on R²:", best_model_name)

## 17. Actual vs Predicted Auction Prices

In [ ]:
plt.figure(figsize=(7,7))

plt.scatter(y_test, best_predictions, alpha=0.7)

minimum = min(y_test.min(), best_predictions.min())
maximum = max(y_test.max(), best_predictions.max())

plt.plot([minimum, maximum], [minimum, maximum])

plt.xlabel("Actual Sold Price")
plt.ylabel("Predicted Sold Price")
plt.title("Actual vs Predicted IPL Auction Price")
plt.show()

## 18. Residual Analysis

In [ ]:
residuals = y_test - best_predictions

plt.figure(figsize=(8,5))
plt.scatter(best_predictions, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")

plt.xlabel("Predicted Sold Price")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted Auction Price")
plt.show()

## 19. Ridge coefficient analysis

Because the predictors were standardized, the absolute Ridge coefficients can be compared to identify variables with stronger associations with the predicted log auction price.

This is an interpretation of the model, not proof of causation.

In [ ]:
ridge_coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": ridge_model.coef_
})

ridge_coefficients["Absolute Coefficient"] = ridge_coefficients["Coefficient"].abs()

ridge_coefficients = ridge_coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
)

display(ridge_coefficients.head(15))

In [ ]:
top_coefficients = ridge_coefficients.head(10).sort_values("Absolute Coefficient")

plt.figure(figsize=(9,6))
plt.barh(
    top_coefficients["Feature"],
    top_coefficients["Absolute Coefficient"]
)

plt.xlabel("Absolute Standardized Ridge Coefficient")
plt.ylabel("Feature")
plt.title("Top Features Associated with Auction Price")
plt.tight_layout()
plt.show()

# 20. Final Conclusion

The project predicts IPL player auction prices using player characteristics and cricket performance statistics.

The improved pipeline addresses:
- Missing values
- Categorical variables
- Extreme predictor values through IQR-based capping
- Right-skewed auction prices through log transformation
- Multicollinearity and coefficient instability through Ridge regularization

The final model should be selected based on the test-set performance of MAE, RMSE and R².

**Use the actual scores generated by this notebook when writing the final resume.**